# Preprocessing: Quantilemaps

This is the final step of the preprocessing of the raw indicator data, from regional aggregates/simulations to quantilemaps. Quantilemaps are nc files that have quantiles, warming levels, and latitudes and longitudes or region names as dimensions and indicator values as datavariables. They are later used to obtain emulations in conjunction with the GMT data for the scenario from an SCM (if you want to use how to obtain that from the SCM FaIR follow 005_run_FaIR.ipynb). 

Specifically, we will create in this tutorial quantile maps for the heating-degree-days indicator per grid point and per region using the aggregated data from the previous tutorial (`002_preprocessing_regional_aggregation.ipynb`).

First we need to (re-)define our indicator and paths, as done in `001_preprocessing_data_setup.ipynb` and `002_preprocessing_regional_aggregation.ipynb`. Usually you'd set these directly in `config.toml`; we set them here in code instead, purely so the tutorial is self-contained and doesn't require editing the config file.

In [6]:
import xarray as xr
import pandas as pd
from rimeX.config import CONFIG 

CONFIG['isimip.download_folder'] = 'data/downloads'
CONFIG['indicators.folder'] = 'data/indicators'
CONFIG['isimip.climate_impact_explorer'] = 'data'
CONFIG['indicator.heating-degree-days'] = {
    'frequency': 'annual',
    'units': 'days*°C',
    'isimip_meta': {
        'db_file': 'data/heating-degree-days.json'
    }
}
CONFIG['preprocessing.regional.weights'] = ["latWeight"]
CONFIG['preprocessing.regional.masks_folder'] = 'data/continental_masks'

Now we define the config specific to this step: the warming level file path — a path to a CSV that records, for every year of every (model, experiment) simulation, which global warming level (GWL) that year corresponds to.

The file looks like this:

```
model,experiment,warming_level,year
CanESM5,ssp126,0.0,1860
CanESM5,ssp126,0.0,1861
CanESM5,ssp126,0.0,1862
...
```

Each row maps one `(model, experiment, year)` combination to a `warming_level` (e.g. `0.0`, `1.5`, `2.0`, ...°C above pre-industrial). This is the bridge between calendar time and warming level: rather than binning your indicator by year, RIME-X uses this file to bin it by the warming level that year corresponds to for that specific model/experiment -- which is what allows results from different models and scenarios (which reach a given warming level in different years) to be pooled together in the quantile maps.

We've included a ready-made warming level file covering all primary and secondary ISIMIP3b scenario simulations that are typically used as source data for RIME-X, so in most cases you can just point to it and move on. If you're working with a different set of simulations (e.g. a different model, scenario, or simulation round not covered by ours), you'll need to derive the warming levels yourself from the corresponding `tas` (near-surface air temperature) simulations, and save them in exactly this CSV format for RIME-X to pick up.

In [7]:
warming_level_file = 'data/warming_levels_ISIMIP.csv'

## Running the quantilemap creation

`make_quantilemaps` (built on top of `make_quantile_map_array`) computes, for a given indicator, a **quantile map**: instead of a value per calendar year, it aggregates the indicator across all models and scenarios by **global warming level (GWL)**, using the warming-level file (`warming_levels_ISIMIP.csv`) to look up which years of which simulations correspond to which warming level. For each warming level, it pools the corresponding years across all models/scenarios and computes a distribution of quantiles -- this is what lets RIME-X later emulate the indicator at *any* GWL of interest, independent of which specific year or scenario originally reached it.

```python
make_quantilemaps(
    indicator,                        # str or list[str]: indicator name(s) to process
    warming_level_file=None,          # path to the warming levels CSV (default: auto-resolved, see below)
    regional=False,                   # one output file per region, including admin boundaries
    regional_no_admin=False,          # one merged output file across all regions, no admin boundaries
    map=False,                        # gridded lat/lon quantile maps (for non-regionally-aggregated indicators)
    weight="latWeight",               # regional weighting scheme to use (for regional/regional_no_admin)
    region=None,                      # regions to process, for regional=True (default: all regions)
    quantile_bins=None,               # number of quantile bins (default: preprocessing.quantilemap_quantile_bins)
    running_mean_window=None,         # years to smooth over before sampling each warming level (default: preprocessing.running_mean_window)
    projection_baseline=None,         # baseline period for the indicator's transform (default: preprocessing.projection_baseline)
    equiprobable_climate_models=True, # give every model equal weight in each warming-level bin, regardless of how often it appears
    overwrite=False,                  # recompute even if output files already exist
    season=None,                      # season(s) to process, e.g. "summer" (default: all seasons in preprocessing.seasons)
    simulation_round=None,            # ISIMIP simulation round(s) to restrict to, e.g. ["ISIMIP3b"] (default: isimip.simulation_round)
    skip_transform=False,             # use the indicator's absolute values instead of its baseline-relative transform
    warming_levels=None,              # subset of warming levels to process, e.g. [1.5, 2.0, 3.0] (default: all levels in the file)
    map_chunk_size=None,              # process lat/lon maps in latitude chunks of this size, to limit memory usage
    suffix="",                        # custom suffix appended to output filenames
    auto_suffix=True,                 # auto-generate a suffix reflecting non-default options used (e.g. "_eq", "_rmw5", "_qb51")
    skip_nans=False,                  # skip NaN values when computing quantiles, rather than propagating them
    cpus=None,                        # number of parallel worker processes (across regions/output files)
)
```

**On `warming_level_file` and output location:** the folder that quantile maps get written to is always derived from wherever the warming-level file lives (its parent directory becomes the output root). If you pass your own `warming_level_file` (as we do here, pointing at `data/warming_levels_ISIMIP.csv`), output lands directly under `data/`. If you leave `warming_level_file=None`, RIME-X auto-resolves it to:

```
{CONFIG['isimip.climate_impact_explorer']}/isimip3/running-{running_mean_window}-years/warming_levels.csv
```

For that auto-resolution to work, a warming-level file needs to already exist at that exact path (built with the running-mean window you intend to use, since it's baked into the folder name) -- otherwise you'll need to generate one there first, or just pass `warming_level_file` explicitly as we do below.

**Outcome:** where the regional-average step gave you one CSV per (model, scenario) with a `time` column, this step collapses across *all* models and scenarios and reorganizes the data by warming level instead of by calendar time -- producing one NetCDF file with dimensions `(warming_level, quantile, region)` containing the indicator's quantile distribution at each warming level. This is the file the emulator reads from in later steps.

**From the command line:** the same computation is available as a CLI tool once your indicators and warming-level file are set in `config.toml`, e.g.:

```bash
rime-preproc-quantilemaps -i heating-degree-days --regional --weight latWeight --warming-level-file data/warming_levels_ISIMIP.csv
```

This accepts the same options as above (`--regional-no-admin`, `--map`, `--quantile-bins`, `--running-mean-window`, `--overwrite`, etc.), so you can run this step outside a notebook once your config is finalized.

In the following, we create the quantile maps per aggregated region:

In [8]:
from rimeX.preproc.quantilemaps import make_quantilemaps
make_quantilemaps(indicator=["heating-degree-days"], regional = True, warming_level_file = warming_level_file, weight="latWeight")

[14:03:55 | rimeX | INFO] Collect quantile maps data for heating-degree-days | annual. Warming levels 0.0 to 6.9
100%|██████████| 16/16 [00:02<00:00,  7.72it/s]
[14:03:57 | rimeX | INFO] Compute quantiles for collected data heating-degree-days | annual.
100%|██████████| 66/66 [00:01<00:00, 58.39it/s]
[14:03:58 | rimeX | INFO] Write to data/quantilemaps_regional_admin/heating-degree-days/AFR/heating-degree-days_annual_afr_latweight_eq.nc
[14:03:58 | rimeX | INFO] Collect quantile maps data for heating-degree-days | annual. Warming levels 0.0 to 6.9
100%|██████████| 16/16 [00:01<00:00,  8.11it/s]
[14:04:00 | rimeX | INFO] Compute quantiles for collected data heating-degree-days | annual.
100%|██████████| 66/66 [00:01<00:00, 58.67it/s]
[14:04:02 | rimeX | INFO] Write to data/quantilemaps_regional_admin/heating-degree-days/ANT/heating-degree-days_annual_ant_latweight_eq.nc
[14:04:02 | rimeX | INFO] Collect quantile maps data for heating-degree-days | annual. Warming levels 0.0 to 6.9
100%|

[PosixPath('data/quantilemaps_regional_admin/heating-degree-days/AFR/heating-degree-days_annual_afr_latweight_eq.nc'),
 PosixPath('data/quantilemaps_regional_admin/heating-degree-days/ANT/heating-degree-days_annual_ant_latweight_eq.nc'),
 PosixPath('data/quantilemaps_regional_admin/heating-degree-days/ASI/heating-degree-days_annual_asi_latweight_eq.nc'),
 PosixPath('data/quantilemaps_regional_admin/heating-degree-days/EUR/heating-degree-days_annual_eur_latweight_eq.nc'),
 PosixPath('data/quantilemaps_regional_admin/heating-degree-days/NOR/heating-degree-days_annual_nor_latweight_eq.nc'),
 PosixPath('data/quantilemaps_regional_admin/heating-degree-days/OCE/heating-degree-days_annual_oce_latweight_eq.nc'),
 PosixPath('data/quantilemaps_regional_admin/heating-degree-days/SEV/heating-degree-days_annual_sev_latweight_eq.nc'),
 PosixPath('data/quantilemaps_regional_admin/heating-degree-days/SOU/heating-degree-days_annual_sou_latweight_eq.nc')]

Here we create the quantilemaps per gridpoint: 

In [9]:
make_quantilemaps(indicator=["heating-degree-days"], map = True, warming_level_file = warming_level_file)

[14:04:26 | rimeX | INFO] Collect quantile maps data for heating-degree-days | annual. Warming levels 0.0 to 6.9
100%|██████████| 16/16 [00:03<00:00,  5.13it/s]
[14:04:30 | rimeX | INFO] Compute quantiles for collected data heating-degree-days | annual.
100%|██████████| 66/66 [00:03<00:00, 19.63it/s]
[14:04:33 | rimeX | INFO] Write to data/quantilemaps/heating-degree-days/heating-degree-days_annual_quantilemaps_eq.nc


[PosixPath('data/quantilemaps/heating-degree-days/heating-degree-days_annual_quantilemaps_eq.nc')]

## Next steps

At this point, `heating-degree-days` has quantile maps computed per region (with admin boundaries), ready for emulation. See `004_emulations.ipynb` to run emulations using these quantile maps together with GMT (global mean temperature) data for your scenarios of interest. If you still need to obtain GMT data for your scenarios of interest from an SCM (Simple Climate Model), see `005_run_FaIR.ipynb` first.

## Appendix: additional quantilemap config options

Beyond the `make_quantilemaps` options already covered above, a few more settings live in `config.toml` itself and are worth knowing about (see `[preprocessing]`/`[emulator]` in `config.toml` for the underlying values).

| Key | Type | Description |
|---|---|---|
| `preprocessing.running_mean_window` | int | Default smoothing window (in years) applied before sampling each warming level. `make_quantilemaps`'s `running_mean_window` argument overrides this per call. |
| `preprocessing.quantilemap_quantile_bins` | int | Default number of quantile bins computed per warming level (e.g. `101` for percentile-level resolution). Overridden via `quantile_bins`. |
| `preprocessing.projection_baseline` | [int, int] | Default baseline period used for the indicator's `baseline_change`/`baseline_change_percent` transform. Overridden via `projection_baseline`. |
| `preprocessing.warming_level_step`, `warming_level_min` | float | Used when *generating* a warming-level file from scratch (not needed if you already have one, like `warming_levels_ISIMIP.csv`). |
| `preprocessing.seasons` | dict[str, list[int]] | Named seasons (`annual`, `winter`, `summer`, ...) as lists of months; controls which months are averaged for a given `season`. |
| `emulator.quantiles` | list[float] | The quantiles the *emulator* will later sample from a quantile map (e.g. `[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]`) -- distinct from `quantile_bins`, which controls how finely the quantile map itself is stored. |
| `emulator.warming_level_step` | float | Step size used when interpolating between warming levels at emulation time. |

For most use cases, the defaults are reasonable -- the running mean window, quantile bin count, and baseline period only need adjusting for specialized indicators, and season/simulation-round filtering mainly matters once you're working with sub-annual or multi-round data.